# Progressive Projection

## Dependencies

In [2]:
import os
import sys
import numpy as np
import rasterio
import rasterio.mask
from rasterio.merge import merge
from rasterio.io import MemoryFile
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject, transform_bounds
from shapely.geometry import box
from dbfread import DBF
import pandas as pd
import tempfile
import subprocess
from plyfile import PlyData, PlyElement
import trimesh
import pyvista as pv
from ipywidgets import interact, Dropdown

## DEM Pre-processing

In this step, we get our DEM(s) together and into the right CRS. If you plan on downsampling, it is recommended you output both a full res and downsampled version as full res is necessary for effective texture shading.

In [8]:
# --- PARAMETERS ---
input_paths = ["../data/wurl/USGS_13_n41w112_20260519.tif"]  # Single file or list of files
output_path = "../data/wurl/processed_dem_downsampled.tif"
target_crs = "EPSG:32143"  # Your desired output CRS
downsample_factor = 2.0  # 1.0 to skip

opened_files = []
try:
    # 1. Mosaic / Load
    if len(input_paths) == 1:
        src = rasterio.open(input_paths[0])
        opened_files.append(src)
    else:
        opened_files = [rasterio.open(p) for p in input_paths]
        mosaic_array, mosaic_transform = merge(opened_files)
        
        profile = opened_files[0].profile.copy()
        profile.update({"height": mosaic_array.shape[1], "width": mosaic_array.shape[2], "transform": mosaic_transform})
        
        memfile = MemoryFile()
        src = memfile.open(**profile)
        src.write(mosaic_array)
        opened_files.extend([memfile, src])

    # 2. Downsample
    if downsample_factor != 1.0:
        new_h, new_w = int(src.height / downsample_factor), int(src.width / downsample_factor)
        ds_transform = src.transform * src.transform.scale((src.width / new_w), (src.height / new_h))
        ds_data = src.read(out_shape=(src.count, new_h, new_w), resampling=Resampling.bilinear)
        
        profile = src.profile.copy()
        profile.update({"height": new_h, "width": new_w, "transform": ds_transform})
        
        memfile_ds = MemoryFile()
        src = memfile_ds.open(**profile)
        src.write(ds_data)
        opened_files.extend([memfile_ds, src])

    # 3. Reproject
    dst_crs = rasterio.crs.CRS.from_string(target_crs)
    transform, width, height = calculate_default_transform(src.crs, dst_crs, src.width, src.height, *src.bounds)
    
    profile = src.profile.copy()
    profile.update({'crs': dst_crs, 'transform': transform, 'width': width, 'height': height})
    
    memfile_repr = MemoryFile()
    src_repr = memfile_repr.open(**profile)
    for i in range(1, src.count + 1):
        reproject(rasterio.band(src, i), rasterio.band(src_repr, i), src_transform=src.transform, 
                  src_crs=src.crs, dst_transform=transform, dst_crs=dst_crs, resampling=Resampling.bilinear)
    opened_files.extend([memfile_repr, src_repr])

    # 4. Crop NoData Edges
    band = src_repr.read(1)
    nodata = src_repr.nodata if src_repr.nodata is not None else 0
    valid_mask = (band != nodata)
    
    rows = np.any(valid_mask, axis=1)
    cols = np.any(valid_mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    
    left, top = src_repr.transform * (cmin, rmin)
    right, bottom = src_repr.transform * (cmax + 1, rmax + 1)
    valid_bbox = box(left, bottom, right, top)
    
    # FIX: Explicit reference using module scope ensures notebook safely executes every time
    cropped_data, cropped_transform = rasterio.mask.mask(src_repr, [valid_bbox.__geo_interface__], crop=True)

    # 5. Save Output
    final_profile = src_repr.profile.copy()
    final_profile.update({"height": cropped_data.shape[1], "width": cropped_data.shape[2], "transform": cropped_transform})
    
    with rasterio.open(output_path, "w", **final_profile) as dst:
        dst.write(cropped_data)
    print("Processing completed successfully.")

finally:
    for f in reversed(opened_files):
        try: f.close()
        except: pass

Processing completed successfully.


## Gather Accompanying Layers

### Texture Shading

In [13]:
# --- PARAMETERS ---
dem_path = "../data/wurl/processed_dem.tif"
# Optional: Path to your downsampled DEM target. Set to None if you want full resolution.
downsampled_target_path = "../data/wurl/processed_dem_downsampled.tif" 
output_path = "../data/wurl/texture_shade.tif"

detail = 75        # 0-200: 0=raw DEM, 50=standard terrain, 75=complex terrain, 200=edge detection
contrast = 1.0

# 1. Load the Full-Resolution DEM
with rasterio.open(dem_path) as src:
    dem = src.read(1).astype(np.float64)
    profile = src.profile
    nodata = src.nodata

# Handle NoData on full res
if nodata is not None:
    mask = dem == nodata
    dem[mask] = np.nan

mean_elev = np.nanmean(dem)
dem_filled = np.where(np.isnan(dem), mean_elev, dem)

# 2. 2D FFT and frequency filtering (Processes on FULL resolution)
fft_shift = np.fft.fftshift(np.fft.fft2(dem_filled))
rows, cols = dem.shape
cy, cx = rows // 2, cols // 2

y, x = np.arange(rows) - cy, np.arange(cols) - cx
xx, yy = np.meshgrid(x, y)
freq = np.sqrt(xx**2 + yy**2)
freq[cy, cx] = 1  # avoid DC divide-by-zero

filt = np.power(freq, detail / 100.0)
filt[cy, cx] = 0  # zero DC component

fft_filtered = fft_shift * filt
result = np.real(np.fft.ifft2(np.fft.ifftshift(fft_filtered))) * contrast

# 3. Controlled Percentile Stretch (Safe 8-bit mapping)
# Calculate high-contrast percentiles to ignore outlier spikes/depressions
lo, hi = np.nanpercentile(result, 2), np.nanpercentile(result, 98)

if lo == hi:
    result_norm = np.zeros_like(result)
else:
    # Scale the values strictly between 1 and 255. 
    # This prevents extreme values from blowing out contrast, but guarantees nothing becomes 0.
    result_norm = 1 + ((result - lo) / (hi - lo) * 254)

# Clamp everything safely inside 1 to 255 so the valley floors stay visible
result_uint8 = np.clip(result_norm, 1, 255).astype(np.uint8)

# Force ONLY the true outer geographic boundary mask to 0 (NoData)
if nodata is not None:
    result_uint8[mask] = 0

# 4. Optional Downsampling step
if downsampled_target_path is not None:
    print("Downsampled target provided. Resampling final texture shade...")
    
    # Read the metadata/specs of your downsampled target DEM
    with rasterio.open(downsampled_target_path) as target_src:
        target_profile = target_src.profile
        target_width = target_src.width
        target_height = target_src.height
    
    # Create an empty destination array matching the target dimensions
    resampled_result = np.zeros((target_height, target_width), dtype=np.uint8)
    
    # Use Rasterio's virtual reproject/resample tool to scale down the image cleanly
    rasterio.warp.reproject(
        source=result_uint8,
        destination=resampled_result,
        src_transform=profile['transform'],
        src_crs=profile['crs'],
        dst_transform=target_profile['transform'],
        dst_crs=target_profile['crs'],
        resampling=Resampling.bilinear  # Smoothly blends pixel textures while shrinking
    )
    
    # Update profile to match the downsampled target properties, forcing nodata to 0
    profile.update(
        dtype=rasterio.uint8,
        count=1,
        nodata=0,  
        width=target_width,
        height=target_height,
        transform=target_profile['transform'],
        crs=target_profile['crs']
    )
    output_data = resampled_result
else:
    # If no target provided, fallback to original full-res profile
    profile.update(dtype=rasterio.uint8, count=1, nodata=0)
    output_data = result_uint8

# 5. Export the file
with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(output_data, 1)

print(f"High-contrast 8-bit Texture shade layer created perfectly matching target: {output_path}")

Downsampled target provided. Resampling final texture shade...
High-contrast 8-bit Texture shade layer created perfectly matching target: ../data/wurl/texture_shade.tif


### Fractional Cover

We want 0-1 values expressing the average percent total vegetation cover of our pixels. GEE does this well. Use [this script](https://code.earthengine.google.com/a9da62bc80f967590ef2e7a99dae07a6).

### Land Cover

NLCD is perfectly fine here. I actually like the NLCD natural classes, but I like to make maps that have as little evidence of human development as possible. So instead, I use LANDFIRE BPS, which simulates Pre-European conditions. I often map those BPS classes to NLCD's classification scheme for simplicity.

In [19]:
# 1. Define the path to your local DEM file
dem_path = "../data/wurl/processed_dem_downsampled.tif"

with rasterio.open(dem_path) as src:
    # --- Area of Interest (AOI) ---
    # Extract bounding box in the DEM's native CRS
    left, bottom, right, top = src.bounds
    
    # Transform bounds to WGS84 (EPSG:4326) as required by LANDFIRE
    wgs84_bounds = transform_bounds(src.crs, 'EPSG:4326', left, bottom, right, top)
    
    # Format as space-separated coordinates in W, S, E, N order
    aoi_wgs84 = f"{wgs84_bounds[0]} {wgs84_bounds[1]} {wgs84_bounds[2]} {wgs84_bounds[3]}"
    
    # --- Output Projection ---
    # Try to get the Well-Known ID (EPSG code). If it's a custom/complex CRS, fallback to WKT.
    if src.crs.is_epsg_code:
        output_projection = str(src.crs.to_epsg())
    else:
        output_projection = src.crs.to_wkt()
        
    # --- Resample Resolution ---
    # Extract cell size (resolution). Assuming square pixels, we take the absolute value of the X resolution.
    # LANDFIRE requires an integer between 30 and 9999.
    res_x = abs(src.res[0])
    resample_resolution = int(round(res_x))
    
    # Ensure it falls within the valid range (defaulting to 30 if smaller)
    if resample_resolution < 30:
        print(f"Warning: Calculated resolution is {resample_resolution}m, but LANDFIRE minimum is 30m. Setting to 30.")
        resample_resolution = 30
    elif resample_resolution > 9999:
        resample_resolution = 9999

# --- Print the payload for your request ---
print("--- LANDFIRE Request Fields ---")
print(f"Layer List:         LF2020_BPS")
print(f"Area of Interest:   {aoi_wgs84}")
print(f"Output Projection:  {output_projection}")
print(f"Resample Resolution:{resample_resolution}")

--- LANDFIRE Request Fields ---
Layer List:         LF2020_BPS
Area of Interest:   -112.00794891796943 39.998287724348934 -110.99204533900156 41.00161667309758
Output Projection:  32143
Resample Resolution:30


Go to https://lfps.usgs.gov/ and use the above information to request the data.

#### Optional Remapping

In [11]:
# Path to your attribute table
bps_tif_path = "../data/wurl/j8d65659812364e8ca1217f3ac29499ce.tif"
bps_dbf_path = bps_tif_path + ".vat.dbf" if os.path.exists(bps_tif_path + ".vat.dbf") else bps_tif_path.replace(".tif", ".dbf")

# 2. Output path for the new NLCD raster
output_nlcd_tif = "../data/wurl/nlcd_reclassified_from_bps.tif"

# OPTIONAL: Set this to a path (e.g., "your_dem.tif") to force the output 
# to perfectly match its dimensions, extent, and CRS. Set to None to skip.
template_tif = "../data/wurl/processed_dem_downsampled.tif" 

# ==========================================
# 1. READ ATTRIBUTES & BUILD DICTIONARY
# ==========================================
dbf = DBF(bps_dbf_path, load=True)
df_attr = pd.DataFrame(dbf.records)

groupveg_to_nlcd_code = {
    'Open Water': 11,
    'Barren-Rock/Sand/Clay': 31,
    'Conifer': 42,
    'Hardwood': 41,
    'Shrubland': 52,
    'Grassland': 71,
    'Riparian': 90,
    'Savanna': 52,
    'Sparsely Vegetated': 31,
    'Herbaceous / Non-vascular': 71
}

def get_nlcd_code(groupveg_str):
    if not isinstance(groupveg_str, str): return 0
    clean_str = groupveg_str.strip()
    if clean_str in groupveg_to_nlcd_code: return groupveg_to_nlcd_code[clean_str]
    
    clean_lower = clean_str.lower()
    if 'shrub' in clean_lower or 'sagebrush' in clean_lower: return 52
    if 'conifer' in clean_lower: return 42
    if 'hardwood' in clean_lower or 'aspen' in clean_lower: return 41
    if 'grass' in clean_lower or 'meadow' in clean_lower: return 71
    if 'riparian' in clean_lower or 'wetland' in clean_lower: return 90
    if 'barren' in clean_lower or 'sparse' in clean_lower: return 31
    return 0

df_attr['NLCD_CODE'] = df_attr['GROUPVEG'].apply(get_nlcd_code)
mapping_dict = dict(zip(df_attr['Value'], df_attr['NLCD_CODE']))

# ==========================================
# 2. READ RASTER & RECLASSIFY IN-MEMORY
# ==========================================
print(f"Reading original raster: {bps_tif_path}")
with rasterio.open(bps_tif_path) as src:
    src_profile = src.profile.copy()
    src_crs = src.crs
    src_transform = src.transform
    raster_data = src.read(1)
    
    print("Mapping pixel values to NLCD classes...")
    reclassified_raster = np.zeros_like(raster_data, dtype=np.uint8)
    for bps_val, nlcd_code in mapping_dict.items():
        reclassified_raster[raster_data == bps_val] = nlcd_code
        
    if src.nodata is not None:
        reclassified_raster[raster_data == src.nodata] = 0

# ==========================================
# 3. WRITE OUT FILE (WITH OPTIONAL TEMPLATE)
# ==========================================
if template_tif and os.path.exists(template_tif):
    print(f"Template detected. Aligning and resampling output to match: {template_tif}")
    
    with rasterio.open(template_tif) as template:
        # Extract target spatial configurations
        dst_crs = template.crs
        dst_transform = template.transform
        dst_width = template.width
        dst_height = template.height
        
        # Build destination profile matching the template structure
        dst_profile = template.profile.copy()
        dst_profile.update(
            dtype=rasterio.uint8,
            count=1,
            nodata=0
        )
        
    # Allocate empty array matching target dimensions
    resampled_raster = np.zeros((dst_height, dst_width), dtype=np.uint8)
    
    # Reproject/Resample the reclassified grid into the template framework
    # Resampling.nearest is critical here to preserve integer categorical codes!
    reproject(
        source=reclassified_raster,
        destination=resampled_raster,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest
    )
    
    print(f"Writing resampled raster to: {output_nlcd_tif}")
    with rasterio.open(output_nlcd_tif, 'w', **dst_profile) as dst:
        dst.write(resampled_raster, 1)

else:
    if template_tif:
        print(f"Warning: Template path '{template_tif}' not found. Defaulting to original layout.")
        
    print(f"Writing native raster to: {output_nlcd_tif}")
    src_profile.update(dtype=rasterio.uint8, count=1, nodata=0)
    with rasterio.open(output_nlcd_tif, 'w', **src_profile) as dst:
        dst.write(reclassified_raster, 1)

print("Finished successfully!")

Reading original raster: ../data/wurl/j8d65659812364e8ca1217f3ac29499ce.tif
Mapping pixel values to NLCD classes...
Template detected. Aligning and resampling output to match: ../data/wurl/processed_dem_downsampled.tif
Writing resampled raster to: ../data/wurl/nlcd_reclassified_from_bps.tif
Finished successfully!


### Bare Earth Color

Go to https://www.nrcs.usda.gov/resources/education-and-teaching-materials/soil-colors-of-the-united-states and download the approptiate state file(s) for your project

In [2]:
# --- PARAMETERS ---
template_path = "../data/wurl/processed_dem_downsampled.tif"
source_path = "../data/wurl/UT-soil-color-125cm-gNATSGO-highres.tif"
output_path = "../data/wurl/aligned_soil_raster.tif"

# --- PROCESSING ---
with rasterio.open(template_path) as template:
    # Extract the exact target profile metrics from your template TIF
    target_profile = template.profile.copy()
    target_crs = template.crs
    target_transform = template.transform
    target_width = template.width
    target_height = template.height
    
    # Ensure the output profile matches your template's data layout
    # Classification rasters typically use integers (e.g., uint8 or uint16)
    # We read the data type directly from your source class raster
    with rasterio.open(source_path) as source:
        source_dtype = source.dtypes[0]
        source_nodata = source.nodata if source.nodata is not None else 0
        
        target_profile.update({
            'dtype': source_dtype,
            'nodata': source_nodata,
            'count': 1
        })

        # Create the output file with the template's exact configurations
        with rasterio.open(output_path, 'w', **target_profile) as dst:
            
            print(f"Resampling categorical raster: {source_path}...")
            print(f"Target dimensions: {target_width}x{target_height}")
            print(f"Data Type preserved: {source_dtype}")

            # Reproject/Resample the data manually
            reproject(
                source=rasterio.band(source, 1),
                destination=rasterio.band(dst, 1),
                src_transform=source.transform,
                src_crs=source.crs,
                dst_transform=target_transform,
                dst_crs=target_crs,
                resampling=Resampling.nearest,
                dst_nodata=source_nodata
            )
            
print(f"SUCCESS: Pixel-perfect aligned class file written to {output_path}")

Resampling categorical raster: ../data/wurl/UT-soil-color-125cm-gNATSGO-highres.tif...
Target dimensions: 4673x6084
Data Type preserved: uint16
SUCCESS: Pixel-perfect aligned class file written to ../data/wurl/aligned_soil_raster.tif


## Create `.ply`

In [8]:
# --- PARAMETERS ---
DEM_TIF_PATH = "../data/wurl/processed_dem_downsampled.tif"
OUTPUT_PLY_PATH = "../data/wurl/base.ply"

# --- HELPER FUNCTIONS ---
def write_flat_ply(path, vertices_xyz, faces_0idx):
    """Writes a clean binary little-endian PLY file of the unwarped terrain."""
    n_verts = len(vertices_xyz)
    n_faces = len(faces_0idx)

    header = (
        f"ply\n"
        f"format binary_little_endian 1.0\n"
        f"element vertex {n_verts}\n"
        f"property float x\nproperty float y\nproperty float z\n"
        f"element face {n_faces}\n"
        f"property list uchar uint vertex_indices\n"
        f"end_header\n"
    ).encode("ascii")

    # Pack structure for coordinates
    vert_dtype = np.dtype([("x", "<f4"), ("y", "<f4"), ("z", "<f4")])
    vert_struct = np.empty(n_verts, dtype=vert_dtype)
    vert_struct["x"] = vertices_xyz[:, 0]
    vert_struct["y"] = vertices_xyz[:, 1]
    vert_struct["z"] = vertices_xyz[:, 2]

    # Pack structure for quad faces
    face_dtype = np.dtype([("count", "u1"), ("v0", "<u4"), ("v1", "<u4"), ("v2", "<u4"), ("v3", "<u4")])
    face_struct = np.empty(n_faces, dtype=face_dtype)
    face_struct["count"] = 4
    face_struct["v0"] = faces_0idx[:, 0]
    face_struct["v1"] = faces_0idx[:, 1]
    face_struct["v2"] = faces_0idx[:, 2]
    face_struct["v3"] = faces_0idx[:, 3]

    with open(path, "wb") as f:
        f.write(header)
        f.write(vert_struct.tobytes())
        f.write(face_struct.tobytes())
    print(f"\nSuccessfully generated clean terrain sheet: {path}")

# --- PROCESSING ---
print(f"Reading source GeoTIFF dataset: {DEM_TIF_PATH}...")

with rasterio.open(DEM_TIF_PATH) as src:
    out_h = src.height
    out_w = src.width
    
    # Read elevation band directly
    elevation = src.read(1)
    
    # Create the strict validation mask across the full dataset
    nodata_val = src.nodata if src.nodata is not None else -9999
    valid_mask = (elevation != nodata_val) & (elevation > -9000)
    flat_valid = valid_mask.ravel()
    
    # Generate full exact pixel index grids
    cols, rows = np.meshgrid(np.arange(out_w), np.arange(out_h))
    
    # Extract native metric coordinates from the projected CRS
    xs, ys = rasterio.transform.xy(src.transform, rows, cols)
    x_metric = np.array(xs).reshape(out_h, out_w)
    y_metric = np.array(ys).reshape(out_h, out_w)

print(f"Grid size pulled: {out_w}x{out_h} ({out_w * out_h} points)")

# Anchor local space around the center of the full bounding box
x_center = x_metric[out_h // 2, out_w // 2]
y_center = y_metric[out_h // 2, out_w // 2]

x_local = x_metric - x_center
y_local = y_metric - y_center
z_local = elevation * 2.5  # Native elevation exaggeration/scale factor

# Scale geometry down uniformly to fit inside standard scene layouts
SCALE_FACTOR = 0.001 
X_scaled = x_local * SCALE_FACTOR
Y_scaled = y_local * SCALE_FACTOR
Z_scaled = z_local * SCALE_FACTOR

# Compile the 1D structured vertex arrays
flat_raw_pts = np.column_stack((X_scaled.ravel(), Y_scaled.ravel(), Z_scaled.ravel()))

# Build ALL possible quad polygon faces across rows & columns
ri, ci = np.mgrid[0:out_h-1, 0:out_w-1]
tl = ri * out_w + ci
tr = tl + 1
bl = (ri + 1) * out_w + ci
br = bl + 1
all_quads = np.column_stack((tl.ravel(), bl.ravel(), br.ravel(), tr.ravel()))

# Keep only quads where EVERY single corner points to a valid data pixel
face_mask = (flat_valid[all_quads[:, 0]] & 
             flat_valid[all_quads[:, 1]] & 
             flat_valid[all_quads[:, 2]] & 
             flat_valid[all_quads[:, 3]])
kept_quads = all_quads[face_mask]

# Compact: Extract only the unique vertices that are actually used by valid faces
used_verts = np.unique(kept_quads)
compact_pts = flat_raw_pts[used_verts]

# Remap old vertex indices to the new compacted vertex array indices
remap = np.empty(len(flat_raw_pts), dtype=np.int64)
remap[used_verts] = np.arange(len(used_verts))
compact_quads = remap[kept_quads].astype(np.uint32)

# Save output pure surface mesh file
write_flat_ply(OUTPUT_PLY_PATH, compact_pts, compact_quads)

Reading source GeoTIFF dataset: ../data/wurl/processed_dem_downsampled.tif...
Grid size pulled: 4673x6084 (28430532 points)

Successfully generated clean terrain sheet: ../data/wurl/base.ply


## Blender Shadows

In [2]:
# --- PARAMETERS ---
# System Paths
BLENDER_EXE = r"C:\Program Files\Blender Foundation\Blender 5.1\blender.exe"
INPUT_PLY_PATH = "../data/wurl/base.ply"
OUTPUT_PLY_PATH = "../data/wurl/base_shaded.ply"

# Lighting Parameters
SUN_STRENGTH = 5.0          # Intensity of the sun light
SUN_ANGLE_X = 65.0          # Tilt angle in degrees
SUN_ANGLE_Y = 25.0
SUN_ANGLE_Z = 165.0         # Azimuth/Directional angle in degrees

# Material Parameters
MATERIAL_COLOR = (1.0, 1.0, 1.0, 1.0)
MATERIAL_ROUGHNESS = 1.0

# --- GENERATE HEADLESS BLENDER SCRIPT ---
blender_script_content = f"""
import bpy
import os
import mathutils
import array
import math

# --- CONFIGURATION PASSED FROM JUPYTER ---
input_ply = r"{INPUT_PLY_PATH}"
output_ply = r"{OUTPUT_PLY_PATH}"
sun_strength = {SUN_STRENGTH}
sun_angles = ({SUN_ANGLE_X}, {SUN_ANGLE_Y}, {SUN_ANGLE_Z})
mat_color = {MATERIAL_COLOR}
mat_roughness = {MATERIAL_ROUGHNESS}

print("Cleaning up default scene elements...")
bpy.ops.object.select_all(action='SELECT')
bpy.ops.object.delete(use_global=False)

print(f"Importing source PLY file: {{input_ply}}")
bpy.ops.wm.ply_import(filepath=input_ply)

# Grab the newly imported mesh object
terrain_obj = bpy.context.active_object
if not terrain_obj:
    raise ValueError("Failed to import or locate the terrain object.")

terrain_obj.name = "Terrain_Mesh"
mesh = terrain_obj.data
num_loops = len(mesh.loops)
print(f"Loaded mesh with {{num_loops:,}} loop corners.")

# 1. Force Engine Settings to Cycles GPU
scene = bpy.context.scene
scene.render.engine = 'CYCLES'
scene.cycles.device = 'GPU'
scene.cycles.use_adaptive_sampling = False  
scene.cycles.samples = 50                   

# 2. Force Shade Smooth
mesh.polygons.foreach_set("use_smooth", [True] * len(mesh.polygons))
mesh.update()

# 3. Setup Sun Lighting Context
print("Configuring lighting environment...")
light_data = bpy.data.lights.new(name="Sun_Light", type='SUN')
light_data.energy = sun_strength
light_obj = bpy.data.objects.new(name="Sun_Light", object_data=light_data)
scene.collection.objects.link(light_obj)
light_obj.rotation_euler = (
    math.radians(sun_angles[0]), 
    math.radians(sun_angles[1]), 
    math.radians(sun_angles[2])
)

# 4. Setup Material Node Tree for Bake Context
print("Building procedural baking material...")
mat = bpy.data.materials.new(name="Terrain_Bake_Material")
mat.use_nodes = True
nodes = mat.node_tree.nodes
nodes.clear()

# Create Principled BSDF & Output nodes
node_output = nodes.new(type='ShaderNodeOutputMaterial')
node_bsdf = nodes.new(type='ShaderNodeBsdfPrincipled')
node_bsdf.inputs['Base Color'].default_value = mat_color
node_bsdf.inputs['Roughness'].default_value = mat_roughness

mat.node_tree.links.new(node_bsdf.outputs['BSDF'], node_output.inputs['Surface'])
if len(terrain_obj.data.materials) == 0:
    terrain_obj.data.materials.append(mat)
else:
    terrain_obj.data.materials[0] = mat

# 5. Create or clean target attribute layer
attr_name = "shading_pack"
if attr_name not in mesh.attributes:
    color_attr = mesh.attributes.new(name=attr_name, type='BYTE_COLOR', domain='CORNER')
else:
    color_attr = mesh.attributes[attr_name]
mesh.attributes.active = color_attr

# Establish Selection Context
bpy.ops.object.select_all(action='DESELECT')
terrain_obj.select_set(True)
bpy.context.view_layer.objects.active = terrain_obj

# --- BAKE PASS 1: PURE DIFFUSE SHADOWS ---
print("Baking Pass 1: Pure Diffuse Shadows...")
scene.cycles.bake_type = 'DIFFUSE'
scene.render.bake.target = 'VERTEX_COLORS'
scene.render.bake.use_pass_direct = True    
scene.render.bake.use_pass_indirect = True  
scene.render.bake.use_pass_color = False     

bpy.ops.object.bake(type='DIFFUSE')

print("Streaming shadow values to a raw memory buffer...")
shadow_buffer = array.array('f', [0.0]) * (num_loops * 4)
color_attr.data.foreach_get("color", shadow_buffer)

# --- BAKE PASS 2: AMBIENT OCCLUSION ---
print("Baking Pass 2: Ray-Traced Ambient Occlusion...")
scene.cycles.bake_type = 'AO'
scene.render.bake.target = 'VERTEX_COLORS'

# --- DYNAMIC AO DISTANCE CALCULATION ---
world_corners = [terrain_obj.matrix_world @ mathutils.Vector(corner) for corner in terrain_obj.bound_box]
x_co = [c.x for c in world_corners]
z_co = [c.z for c in world_corners]
width_x = max(x_co) - min(x_co)
height_z = max(z_co) - min(z_co)

optimal_distance = height_z * 0.75
if optimal_distance == 0:
    optimal_distance = width_x * 0.05
scene.render.bake.max_ray_distance = optimal_distance

print(f"Terrain Dimensions: Width={{width_x:.2f}}, Height/Relief={{height_z:.2f}}")
print(f"Calculated Optimal AO Ray Distance: {{scene.render.bake.max_ray_distance:.2f}}")

bpy.ops.object.bake(type='AO')

print("Streaming AO values to a raw memory buffer...")
ao_buffer = array.array('f', [0.0]) * (num_loops * 4)
color_attr.data.foreach_get("color", ao_buffer)

# --- LOW-LEVEL MEMORY BLITTING & MERGING ---
print("Merging data streams directly inside C-memory...")
packed_buffer = array.array('f', [0.0]) * (num_loops * 4)

packed_buffer[0::4] = shadow_buffer[0::4] # Shadows -> Red channel
packed_buffer[1::4] = ao_buffer[0::4]     # AO -> Green channel
packed_buffer[2::4] = array.array('f', [0.0]) * num_loops # Clear Blue channel
packed_buffer[3::4] = array.array('f', [1.0]) * num_loops # Full Alpha

print("Writing packed array back to the active mesh data structure...")
color_attr.data.foreach_set("color", packed_buffer)
mesh.update()

# --- EXPORT TO PLY ---
print("Exporting final packed PLY file...")
export_dir = os.path.dirname(output_ply)
if export_dir and not os.path.exists(export_dir):
    os.makedirs(export_dir)

bpy.ops.wm.ply_export(
    filepath=output_ply,
    export_selected_objects=True,
    export_normals=True,
    export_colors='SRGB',
    export_attributes=True
)
print(f"Success! Packed map base saved to: {{output_ply}}")
"""

# --- EXECUTE BLENDER HEADLESS PROCESS ---
with tempfile.NamedTemporaryFile(suffix=".py", delete=False, mode="w") as temp_script:
    temp_script.write(blender_script_content)
    temp_script_path = temp_script.name

try:
    print("Launching headless Blender subprocess background rendering engine...")
    process = subprocess.Popen(
        [BLENDER_EXE, "-b", "-P", temp_script_path],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    
    while True:
        output = process.stdout.readline()
        if output == '' and process.poll() is not None:
            break
        if output:
            print(output.strip())
            
    rc = process.poll()
    if rc != 0:
        print(f"\nERROR: Blender process closed with non-zero exit code ({rc}). Check log output above.")
    else:
        print("\nHeadless processing pipeline completed successfully.")
        
finally:
    if os.path.exists(temp_script_path):
        os.remove(temp_script_path)

Launching headless Blender subprocess background rendering engine...
C:\Users\evane\AppData\Local\Temp\tmpbe1ckhuv.py:59: DeprecationWarning: 'Material.use_nodes' is expected to be removed in Blender 6.0
mat.use_nodes = True
Cleaning up default scene elements...
Importing source PLY file: C:\Users\evane\OneDrive\Desktop\CREATE\Map_Art\data\wurl\base.ply
Loaded mesh with 112,724,440 loop corners.
Configuring lighting environment...
Building procedural baking material...
Baking Pass 1: Pure Diffuse Shadows...
Streaming shadow values to a raw memory buffer...
Baking Pass 2: Ray-Traced Ambient Occlusion...
Terrain Dimensions: Width=85.46, Height/Relief=5.73
Calculated Optimal AO Ray Distance: 4.30
Streaming AO values to a raw memory buffer...
Merging data streams directly inside C-memory...
Writing packed array back to the active mesh data structure...
Exporting final packed PLY file...
Info: File exported successfully
Success! Packed map base saved to: C:\Users\evane\OneDrive\Desktop\CREA

## Texture Shading and FVC Vertex Bake

In [6]:
# --- PARAMETERS ---
INPUT_PLY_PATH = "../data/wurl/base_shaded.ply"
TEXTURE_SHADE_TIF = "../data/wurl/texture_shade.tif"
FRACTIONAL_COVER_TIF = "../data/wurl/fractional_cover.tif"
OUTPUT_PLY_PATH = "../data/wurl/base_shaded_packed.ply"

# --- PROCESSING ---
print("Loading texture shader and fractional cover rasters...")
with rasterio.open(TEXTURE_SHADE_TIF) as src_tex, rasterio.open(FRACTIONAL_COVER_TIF) as src_cov:
    tex_data = src_tex.read(1)
    cov_data = src_cov.read(1)
    img_h, img_w = tex_data.shape

print(f"Reading pre-shaded PLY mesh dataset: {INPUT_PLY_PATH}...")
plydata = PlyData.read(INPUT_PLY_PATH)
vertices = plydata['vertex']

# Read vertex spatial coordinates to calculate exact bounding box
x, y = vertices['x'], vertices['y']
min_x, max_x = x.min(), x.max()
min_y, max_y = y.min(), y.max()

range_x = (max_x - min_x) if max_x != min_x else 1.0
range_y = (max_y - min_y) if max_y != min_y else 1.0

print("Vectorizing vertex mapping directly to standardized image space pixels...")
# Normalize coordinates into 0.0 - 1.0 UV range
u = (x - min_x) / range_x
v = (y - min_y) / range_y

# Convert UV coordinates into precise raster pixel indexes
# Y/North maps inversely to image top-down indexing (1.0 - v)
cols = np.clip((u * (img_w - 1)).astype(np.int32), 0, img_w - 1)
rows = np.clip(((1.0 - v) * (img_h - 1)).astype(np.int32), 0, img_h - 1)

# Direct array index lookup across the whole grid
sampled_blue_channel = tex_data[rows, cols]
sampled_alpha_channel = cov_data[rows, cols]

# FIX: Explicitly convert 0-255 texture data to a 0.0 - 1.0 float range first
# This ensures it acts identically to the FVC data stream before scaling
if tex_data.dtype.kind != 'f':
    sampled_blue_channel = sampled_blue_channel.astype(np.float32) / 255.0

# Scale both float arrays uniformly to 8-bit unsigned integer limits (0-255)
if sampled_blue_channel.max() <= 1.0:
    sampled_blue_channel = (sampled_blue_channel * 255.0).astype(np.uint8)
if sampled_alpha_channel.max() <= 1.0 and sampled_alpha_channel.dtype.kind == 'f':
    sampled_alpha_channel = (sampled_alpha_channel * 255.0).astype(np.uint8)

print("Packing arrays into structured PLY vertex property architecture...")
# Build new structured data type layout from existing properties
dtype_dict = []
for name in vertices.data.dtype.names:
    dtype_dict.append((name, vertices.data.dtype[name]))

# Inject byte properties if the source file was completely missing color headers
if not all(prop in vertices.data.dtype.names for prop in ['red', 'green', 'blue', 'alpha']):
    dtype_dict = [d for d in dtype_dict if d[0] not in ['red', 'green', 'blue', 'alpha']]
    dtype_dict.extend([('red', 'u1'), ('green', 'u1'), ('blue', 'u1'), ('alpha', 'u1')])

# Allocate clean struct array
new_vertex_data = np.empty(vertices.count, dtype=dtype_dict)

# Map all structural spatial elements over
for name in vertices.data.dtype.names:
    if name not in ['red', 'green', 'blue', 'alpha']:
        new_vertex_data[name] = vertices[name]

# PRESERVE: Keep your baked Red (Shadows) and Green (AO) channels untouched
new_vertex_data['red'] = vertices['red']
new_vertex_data['green'] = vertices['green']

# INJECT: Direct overwrite of Blue and Alpha payload elements
new_vertex_data['blue'] = sampled_blue_channel.astype(np.uint8)
new_vertex_data['alpha'] = sampled_alpha_channel.astype(np.uint8)

# Compile elements back to structured layout format
elements_list = [PlyElement.describe(new_vertex_data, 'vertex')]
if 'face' in plydata:
    elements_list.append(plydata['face'])

print(f"Saving final completed asset layout: {OUTPUT_PLY_PATH}...")
PlyData(elements_list, text=False).write(OUTPUT_PLY_PATH)
print("Success! Multi-channel shader pack architecture is fully populated.")

Loading texture shader and fractional cover rasters...
Reading pre-shaded PLY mesh dataset: ../data/wurl/base_shaded.ply...
Vectorizing vertex mapping directly to standardized image space pixels...
Packing arrays into structured PLY vertex property architecture...
Saving final completed asset layout: ../data/wurl/base_shaded_packed.ply...
Success! Multi-channel shader pack architecture is fully populated.


## Land Cover and Bare Earth Color Face Bake

In [7]:
# --- PARAMETERS ---
INPUT_PLY_PATH = "../data/wurl/base_shaded_packed.ply"
LANDCOVER_TIF = "../data/wurl/nlcd_reclassified_from_bps.tif"
SOIL_TIF = "../data/wurl/aligned_soil_raster.tif"
OUTPUT_PLY_PATH = "../data/wurl/base_shaded_packed_faces.ply"

# --- PROCESSING ---
print("Loading landcover and soil integer rasters...")
with rasterio.open(LANDCOVER_TIF) as src_lc, rasterio.open(SOIL_TIF) as src_soil:
    lc_data = src_lc.read(1)
    soil_data = src_soil.read(1)
    img_h, img_w = lc_data.shape

print(f"Reading input PLY mesh dataset: {INPUT_PLY_PATH}...")
plydata = PlyData.read(INPUT_PLY_PATH)
vertices = plydata['vertex']
faces = plydata['face']

# Extract vertex coordinates to compute face centers
x, y = vertices['x'], vertices['y']
min_x, max_x = x.min(), x.max()
min_y, max_y = y.min(), y.max()

range_x = (max_x - min_x) if max_x != min_x else 1.0
range_y = (max_y - min_y) if max_y != min_y else 1.0

print("Calculating face centers to map polygon topology to pixel coordinates...")
# Extract the vertex indices for every quad face
# faces['vertex_indices'] is usually an array of object/lists, explode it to a 2D array
face_v_indices = np.vstack(faces['vertex_indices'])

# Compute the average X and Y coordinate for each face (center point)
face_x_center = np.mean(x[face_v_indices], axis=1)
face_y_center = np.mean(y[face_v_indices], axis=1)

# Normalize the face centers into 0.0 - 1.0 UV range using the same bounding box
u_face = (face_x_center - min_x) / range_x
v_face = (face_y_center - min_y) / range_y

# Map face UVs directly to the image index bounds
cols_face = np.clip((u_face * (img_w - 1)).astype(np.int32), 0, img_w - 1)
rows_face = np.clip(((1.0 - v_face) * (img_h - 1)).astype(np.int32), 0, img_h - 1)

print("Sampling integer values at polygon coordinates...")
# Lookup discrete integer classifications directly from the raster matrices
sampled_landcover = lc_data[rows_face, cols_face].astype(np.int32)
sampled_soil = soil_data[rows_face, cols_face].astype(np.int32)

print("Building new structured PLY face architecture...")
# 1. Define the updated face dtype layout
# It must contain the standard vertex list property, followed by our custom integer properties
face_dtype = [
    ('vertex_indices', 'i4', (4,)),  # Quad face layout: list of 4 integers
    ('landcover', 'i4'),             # Custom face attribute 1
    ('soil_color', 'i4')             # Custom face attribute 2
]

# Allocate clean structured array for the expanded face element list
new_face_data = np.empty(len(faces), dtype=face_dtype)

# Copy the original geometry indices over to the new data structure
new_face_data['vertex_indices'] = face_v_indices

# Inject our sampled integer data arrays directly into the face rows
new_face_data['landcover'] = sampled_landcover
new_face_data['soil_color'] = sampled_soil

# Compile elements into the PLY container list
# We pass vertices down completely untouched, and describe our new faces
elements_list = [
    plydata['vertex'],
    PlyElement.describe(new_face_data, 'face')
]

print(f"Saving final face-attributed asset layout: {OUTPUT_PLY_PATH}...")
PlyData(elements_list, text=False).write(OUTPUT_PLY_PATH)
print("Success! Discrete face architecture classification is locked down.")

Loading landcover and soil integer rasters...
Reading input PLY mesh dataset: ../data/wurl/base_shaded_packed.ply...
Calculating face centers to map polygon topology to pixel coordinates...
Sampling integer values at polygon coordinates...
Building new structured PLY face architecture...
Saving final face-attributed asset layout: ../data/wurl/base_shaded_packed_faces.ply...
Success! Discrete face architecture classification is locked down.
